In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os

current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.append(parent_dir)

from utils.spike_detection import *
from utils.preprocess_neural import *


import matplotlib.pyplot as plt
plt.rcParams["svg.fonttype"] = "none"
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 6,
    "axes.labelsize": 6,
    "axes.titlesize": 6,
    "xtick.labelsize": 5,
    "ytick.labelsize": 5,
    "xtick.major.size": 1.75,
    "ytick.major.size": 1.75,
    "xtick.minor.size": 1.0,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "ytick.minor.size": 1.0,
    "legend.fontsize": 5,
    "axes.linewidth": 0.5,
})
import matplotlib.cm as cm

import numpy.ma as ma
from scipy.ndimage import gaussian_filter
from tqdm import tqdm
import subprocess
import shutil
import pickle
import glob
from utils.GRC_helpers import *

%matplotlib widget


In [2]:
#data_folder = '../data/CKII_pAce38_PX_20251126'
#data_folder = '../data/CKII_pAce21_PR_20250806'
#data_folder = '../data/CKII_pAce45_PX_20260118'
data_folder = '../data/CKII_pAce47_PX_20260128'


# turn data_folder into absolute path
data_folder = os.path.abspath(data_folder)
print(f'Loading data from: {data_folder}')

data_name = os.path.basename(data_folder)
figure_save_folder = f'../figures/{data_name}'
figure_save_folder = os.path.abspath(figure_save_folder)
print(f'Figures will be saved to: {figure_save_folder}')
if not os.path.exists(figure_save_folder):
    os.makedirs(figure_save_folder)

Loading data from: /Volumes/adam-lab/qixin.yang/ClusterCode/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/data/CKII_pAce47_PX_20260128
Figures will be saved to: /Volumes/adam-lab/qixin.yang/ClusterCode/miniVI_Renana_Pipeline/miniVI_PlaceCell_analysis_V4/figures/CKII_pAce47_PX_20260128


In [3]:
# load merged data 

merged_data_path_CS = os.path.join(data_folder, 'merged_aligned_data_CS.pkl')
with open(merged_data_path_CS, 'rb') as f:
    loaded_merged_data_CS = pickle.load(f)
    traces = loaded_merged_data_CS['traces']
    spikes = loaded_merged_data_CS['spikes']
    speed = loaded_merged_data_CS['speed']
    ts_neural = loaded_merged_data_CS['ts_neural']
    x_neural = loaded_merged_data_CS['x_neural']
    y_neural = loaded_merged_data_CS['y_neural']
    hd_angles_neural = loaded_merged_data_CS['hd_angles_neural']
    weights_all = loaded_merged_data_CS['weights_all']
    mean_images = loaded_merged_data_CS['mean_images']
    frame_rate = loaded_merged_data_CS['frame_rate']
    frame_width = loaded_merged_data_CS['frame_width']
    frame_height = loaded_merged_data_CS['frame_height']
    subfolders = loaded_merged_data_CS['subfolders']
    complex_bursts_dicts = loaded_merged_data_CS['complex_bursts_dicts']
    refined_SS = loaded_merged_data_CS['refined_SS']
    all_CS_spikes = loaded_merged_data_CS['all_CS_spikes']
    all_spikes = loaded_merged_data_CS['all_spikes']
    spike_heights_interpolated = loaded_merged_data_CS['spike_heights_interpolated']
    SNR_interpolated = loaded_merged_data_CS['SNR_interpolated']    
    session_start_frames = loaded_merged_data_CS['session_start_frames']
    traces_SNR_interpolated = loaded_merged_data_CS['traces_SNR_interpolated']
    Vm_SNR_interpolated = loaded_merged_data_CS['Vm_SNR_interpolated']
    burst_metrics = loaded_merged_data_CS['burst_metrics']
    plateaus_dicts = loaded_merged_data_CS.get('plateaus_dicts', None)

print(f"Loaded data from {merged_data_path_CS}")
if plateaus_dicts is not None:
    print(f"Loaded plateaus_dicts for {len(plateaus_dicts)} cells")
else:
    print("No plateaus_dicts found in merged data")

UnpicklingError: invalid load key, '\x00'.

In [ ]:
# Optional: Remove bad SNR segments
spikes = all_spikes.copy()
traces = traces_SNR_interpolated
Vms = Vm_SNR_interpolated

SNR_threshold = 3.5
bad_masks = []

# Now for each cell: remove spikes and make traces and Vms NaN where its SNR_interpolated < SNR_threshold
for cell_idx in range(len(spikes)):
    snr_vals = np.asarray(SNR_interpolated[cell_idx])
    if snr_vals.ndim == 0:
        continue
    bad_mask = snr_vals < SNR_threshold
    bad_masks.append(bad_mask)
    if not np.any(bad_mask):
        continue

    traces[cell_idx][bad_mask] = np.nan
    Vms[cell_idx][bad_mask] = np.nan

    def _filter_spikes(spk_arr, bad_mask_local):
        spk_arr = np.asarray(spk_arr, dtype=np.int64)
        if spk_arr.size == 0:
            return spk_arr
        spk_arr = spk_arr[(spk_arr >= 0) & (spk_arr < bad_mask_local.shape[0])]
        return spk_arr[~bad_mask_local[spk_arr]]

    spikes[cell_idx] = _filter_spikes(spikes[cell_idx], bad_mask)
    all_CS_spikes[cell_idx] = _filter_spikes(all_CS_spikes[cell_idx], bad_mask)
    refined_SS[cell_idx] = _filter_spikes(refined_SS[cell_idx], bad_mask)

    print(f'Cell {cell_idx}: removed {np.sum(bad_mask)} bad SNR frames, spikes reduced from {len(loaded_merged_data_CS["all_spikes"][cell_idx])} to {len(spikes[cell_idx])}')

bad_masks = np.array(bad_masks)